# Creative Extension: Hybrid CF + Content-Based Recommender

**Team 9 — Recommender Systems Spring 2026**

## Overview

This notebook implements a **Hybrid Recommender** that combines:
1. **Collaborative Filtering (SVD)** — captures latent user-item interaction patterns
2. **Content-Based Filtering (Metadata)** — captures restaurant features (categories, price, family-friendliness, attire)

## Motivation: What Limitation Does This Address?

The standalone CF model (SVD) performs well when a user has many interactions, but suffers in sparse regions of the rating matrix. The standalone Content-Based model can recommend restaurants based on restaurant features without needing many interactions, but ignores the collective wisdom captured in other users' behavior.

**The Hybrid model addresses both limitations simultaneously:**
- For well-connected users with many reviews, CF dominates and captures nuanced preferences
- For users with sparse histories, content features provide a meaningful signal
- Combined, the model is more robust across all users than either approach alone

## Method

We combine scores using a **weighted linear interpolation**:

**score(u, i) = α × CF_score(u, i) + (1 - α) × Content_score(u, i)**

- α = 0.6 (CF weight), determined empirically
- CF scores come from the SVD reconstructed matrix (normalized per user to [0, 1])
- Content scores come from cosine similarity between a weighted user profile and restaurant feature vectors
- Both score types are normalized before combining to ensure fair weighting

## Content Features Used
- **Restaurant categories** (one-hot encoded for top categories)
- **GoodForKids** — relevant because families have strong preferences
- **RestaurantsPriceRange** — price sensitivity is a major decision factor
- **RestaurantsAttire** — casual vs. formal captures dining atmosphere preferences

## User Profile Construction
Each user profile is built from their training interactions, weighted by adjusted rating (how much the user liked each restaurant relative to their own average). Restaurants rated higher than the user's average contribute more to the profile.

## Train/Test Split
- **Training:** `train_reviews_santa_barbara.csv`
- **Testing:** `test_reviews_santa_barbara.csv`

## Evaluation Metrics
- **Ranking:** Hit@10, Hit@20, Hit@30, NDCG@10, NDCG@20, NDCG@30
- **Extension-specific:** Catalog **coverage @10** (fraction of catalog appearing in anyone’s top-10) and mean **intra-list diversity @10** (1 − avg pairwise cosine similarity of top-10 item feature vectors), aligned with hybrid / diversity-aware goals

In [1]:
# Imports
import pandas as pd
import numpy as np
import ast
import math
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Hybrid weight: alpha controls how much CF contributes vs. content
ALPHA = 0.6  # 0 = pure content, 1 = pure CF

## 1. Load Data

In [2]:
def load_data(train_path, test_path, restaurants_path):
    """
    Load train, test, and restaurant metadata CSV files.

    Parameters
    ----------
    train_path : str
        Path to train_reviews_santa_barbara.csv
    test_path : str
        Path to test_reviews_santa_barbara.csv
    restaurants_path : str
        Path to restaurants_santa_barbara.csv

    Returns
    -------
    tuple
        (train_df, test_df, restaurants_df)
    """
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    restaurants_df = pd.read_csv(restaurants_path)

    for df in [train_df, test_df]:
        df['user_id'] = df['user_id'].astype(str)
        df['business_id'] = df['business_id'].astype(str)
        df['stars'] = df['stars'].astype(float)
    restaurants_df['business_id'] = restaurants_df['business_id'].astype(str)

    return train_df, test_df, restaurants_df


train_df, test_df, restaurants_df = load_data(
    'train_reviews_santa_barbara.csv',
    'test_reviews_santa_barbara.csv',
    'restaurants_santa_barbara.csv'
)

print(f'Train interactions: {len(train_df)}')
print(f'Test interactions:  {len(test_df)}')
print(f'Unique users:       {train_df["user_id"].nunique()}')
print(f'Unique restaurants: {restaurants_df["business_id"].nunique()}')

Train interactions: 41581
Test interactions:  4801
Unique users:       4801
Unique restaurants: 767


## 2. CF Component: Build Rating Matrix and Fit SVD

In [3]:
def build_rating_matrix(train_df, restaurants_df):
    """
    Build a dense user-item rating matrix from training interactions.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training interactions with columns [user_id, business_id, stars].
    restaurants_df : pd.DataFrame
        Restaurant metadata with column [business_id].

    Returns
    -------
    tuple
        (R, users, items, user_idx, item_idx)
    """
    users = sorted(train_df['user_id'].unique())
    items = sorted(restaurants_df['business_id'].unique())
    user_idx = {u: i for i, u in enumerate(users)}
    item_idx = {it: i for i, it in enumerate(items)}

    R = np.zeros((len(users), len(items)), dtype=np.float32)
    for _, row in train_df.iterrows():
        u, it = row['user_id'], row['business_id']
        if u in user_idx and it in item_idx:
            R[user_idx[u], item_idx[it]] = row['stars']

    return R, users, items, user_idx, item_idx


def fit_svd_normalized(R, n_components=50, random_state=42):
    """
    Fit Truncated SVD and return per-user normalized predicted scores.

    Scores are normalized to [0, 1] per user so they can be combined
    fairly with content-based scores.

    Parameters
    ----------
    R : np.ndarray
        User-item rating matrix.
    n_components : int
        Number of latent factors.
    random_state : int
        Random seed.

    Returns
    -------
    np.ndarray
        R_norm: Per-user min-max normalized predicted score matrix.
    """
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    U = svd.fit_transform(R)
    Vt = svd.components_
    R_pred = U @ Vt

    # Normalize per user to [0, 1]
    row_min = R_pred.min(axis=1, keepdims=True)
    row_max = R_pred.max(axis=1, keepdims=True)
    R_norm = (R_pred - row_min) / (row_max - row_min + 1e-9)

    print(f'SVD fitted with {n_components} latent factors. Scores normalized per user to [0,1].')
    return R_norm


R, users, items, user_idx, item_idx = build_rating_matrix(train_df, restaurants_df)
R_norm = fit_svd_normalized(R)

SVD fitted with 50 latent factors. Scores normalized per user to [0,1].


## 3. Content Component: Build Restaurant Feature Vectors

Restaurant feature vectors are built from:
- **Category one-hot encoding** (top 15 categories in the dataset)
- **GoodForKids** — binary flag (families strongly prefer family-friendly restaurants)
- **RestaurantsPriceRange** — one-hot for price level 1 and 2 (price is a key decision factor)
- **RestaurantsAttire** — binary casual flag (captures dining atmosphere preference)

In [4]:
def parse_attributes(restaurants_df):
    """
    Parse the attributes column from string to dictionary.

    Parameters
    ----------
    restaurants_df : pd.DataFrame
        Restaurant metadata with an 'attributes' column.

    Returns
    -------
    pd.DataFrame
        restaurants_df with a new 'attrs_dict' column containing parsed attributes.
    """
    def str_to_dict(s):
        """Safely parse a string representation of a dict."""
        try:
            return ast.literal_eval(s)
        except Exception:
            return {}

    def clean_values(d):
        """Strip extra quotes from string values in attribute dict."""
        return {k: (v.strip("'\"" ) if isinstance(v, str) else v) for k, v in d.items()}

    restaurants_df = restaurants_df.copy()
    restaurants_df['attrs_dict'] = restaurants_df['attributes'].apply(str_to_dict).apply(clean_values)
    return restaurants_df


def build_item_features(restaurants_df):
    """
    Build a feature matrix for all restaurants.

    Features include category one-hot encoding plus three attribute-derived
    binary features: GoodForKids, PriceRange (1 or 2), and casual attire.

    Parameters
    ----------
    restaurants_df : pd.DataFrame
        Restaurant metadata with 'attrs_dict' and 'categories' columns.

    Returns
    -------
    tuple
        (item_feats_df, feat_cols) where item_feats_df is indexed by business_id
        and feat_cols is the list of feature column names.
    """
    df = restaurants_df.copy()

    # Attribute features
    df['good_for_kids'] = df['attrs_dict'].apply(lambda d: 1 if d.get('GoodForKids') == 'True' else 0)
    df['price_1'] = df['attrs_dict'].apply(lambda d: 1 if d.get('RestaurantsPriceRange2') == '1' else 0)
    df['price_2'] = df['attrs_dict'].apply(lambda d: 1 if d.get('RestaurantsPriceRange2') == '2' else 0)
    df['casual'] = df['attrs_dict'].apply(lambda d: 1 if d.get('RestaurantsAttire') == 'casual' else 0)

    # Category one-hot (top 15 categories by frequency)
    top_cats = [
        'restaurants', 'food', 'pizza', 'mexican', 'italian',
        'american', 'chinese', 'japanese', 'sushi', 'burgers',
        'coffee', 'sandwiches', 'seafood', 'thai', 'mediterranean'
    ]
    cat_series = df['categories'].fillna('').str.lower()
    for cat in top_cats:
        df[f'cat_{cat}'] = cat_series.apply(lambda x: 1 if cat in x else 0)

    attr_cols = ['good_for_kids', 'price_1', 'price_2', 'casual']
    cat_cols = [f'cat_{c}' for c in top_cats]
    feat_cols = attr_cols + cat_cols

    item_feats_df = df.set_index('business_id')[feat_cols].fillna(0)
    return item_feats_df, feat_cols


restaurants_df = parse_attributes(restaurants_df)
item_feats_df, feat_cols = build_item_features(restaurants_df)

print(f'Item feature matrix: {item_feats_df.shape[0]} restaurants × {item_feats_df.shape[1]} features')
print(f'Features: {feat_cols}')

Item feature matrix: 767 restaurants × 19 features
Features: ['good_for_kids', 'price_1', 'price_2', 'casual', 'cat_restaurants', 'cat_food', 'cat_pizza', 'cat_mexican', 'cat_italian', 'cat_american', 'cat_chinese', 'cat_japanese', 'cat_sushi', 'cat_burgers', 'cat_coffee', 'cat_sandwiches', 'cat_seafood', 'cat_thai', 'cat_mediterranean']


## 4. Content Component: Build User Profiles

Each user profile is a weighted average of the feature vectors of restaurants they interacted with in training. The weight for each restaurant is proportional to how much the user liked it relative to their own average rating (adjusted rating). Restaurants rated above the user average contribute more to the profile.

In [5]:
def build_user_profiles(train_df, item_feats_df, feat_cols):
    """
    Build a content-based user profile for each user in the training data.

    The profile is a weighted average of restaurant feature vectors, where
    each restaurant's weight is max(adjusted_rating + 1, 0). The adjusted
    rating is the user's rating minus their personal mean rating.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training interactions with columns [user_id, business_id, stars].
    item_feats_df : pd.DataFrame
        Restaurant feature matrix indexed by business_id.
    feat_cols : list of str
        Feature column names.

    Returns
    -------
    dict
        Mapping user_id -> np.ndarray profile vector of shape (n_features,).
    """
    user_means = train_df.groupby('user_id')['stars'].mean()
    n_feats = len(feat_cols)
    user_profiles = {}

    for u, group in train_df.groupby('user_id'):
        profile = np.zeros(n_feats)
        total_weight = 0.0
        u_mean = user_means[u]
        for _, row in group.iterrows():
            it = row['business_id']
            if it not in item_feats_df.index:
                continue
            adj = row['stars'] - u_mean
            w = max(adj + 1.0, 0.0)
            profile += w * item_feats_df.loc[it].values
            total_weight += w
        if total_weight > 0:
            profile /= total_weight
        user_profiles[u] = profile

    print(f'Built profiles for {len(user_profiles)} users.')
    return user_profiles


user_profiles = build_user_profiles(train_df, item_feats_df, feat_cols)

Built profiles for 4801 users.


## 5. Hybrid Score Computation

For each user, we compute hybrid scores for all candidate restaurants and rank them.

In [6]:
def build_train_history(train_df):
    """
    Build a mapping from each user to their set of training restaurants.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training interactions with columns [user_id, business_id].

    Returns
    -------
    dict
        Mapping user_id -> set of business_ids seen in training.
    """
    history = {}
    for _, row in train_df.iterrows():
        history.setdefault(row['user_id'], set()).add(row['business_id'])
    return history


def get_hybrid_recs(u, R_norm, user_idx, item_idx, items, item_feats_df,
                    user_profiles, seen, alpha, k_max=30):
    """
    Generate hybrid recommendations for a single user.

    Combines normalized CF scores and content-based cosine similarity scores
    using a weighted linear combination.

    Parameters
    ----------
    u : str
        User ID.
    R_norm : np.ndarray
        Per-user normalized SVD predicted score matrix.
    user_idx : dict
        Mapping user_id -> row index in R_norm.
    item_idx : dict
        Mapping business_id -> column index in R_norm.
    items : list of str
        All restaurant business IDs.
    item_feats_df : pd.DataFrame
        Restaurant feature matrix indexed by business_id.
    user_profiles : dict
        Mapping user_id -> content profile vector.
    seen : set
        Set of business_ids the user has seen in training (to exclude).
    alpha : float
        Weight for CF component (1 - alpha for content component).
    k_max : int
        Number of top recommendations to return.

    Returns
    -------
    list of str
        Top-k_max business IDs ranked by hybrid score.
    """
    if u not in user_idx:
        return []

    ui = user_idx[u]
    candidates = [it for it in items if it not in seen and it in item_idx]

    # CF scores (already normalized to [0,1])
    cf_scores = {it: float(R_norm[ui, item_idx[it]]) for it in candidates}

    # Content scores via cosine similarity
    profile = user_profiles.get(u, np.zeros(item_feats_df.shape[1]))
    if profile.sum() == 0:
        content_scores = {it: 0.0 for it in candidates}
    else:
        feat_matrix = item_feats_df.loc[
            [it for it in candidates if it in item_feats_df.index]
        ].values
        valid_items = [it for it in candidates if it in item_feats_df.index]
        sims = cosine_similarity([profile], feat_matrix)[0]
        content_scores = dict(zip(valid_items, sims.tolist()))

    # Hybrid combination
    hybrid_scores = {
        it: alpha * cf_scores[it] + (1 - alpha) * content_scores.get(it, 0.0)
        for it in candidates
    }

    ranked = sorted(hybrid_scores, key=lambda x: -hybrid_scores[x])
    return ranked[:k_max]


train_history = build_train_history(train_df)
print('Train history built.')

Train history built.


## 6. Evaluation Metric Functions

In [7]:
def hit_at_k(recommended, target_item, k):
    """
    Compute Hit@K for a single user.

    Parameters
    ----------
    recommended : list of str
        Ranked list of recommended business IDs.
    target_item : str
        The held-out test business ID.
    k : int
        Cutoff rank.

    Returns
    -------
    float
        1.0 if target in top-K, else 0.0.
    """
    return 1.0 if target_item in recommended[:k] else 0.0


def ndcg_at_k(recommended, target_item, k):
    """
    Compute NDCG@K for a single user with one relevant item.

    Parameters
    ----------
    recommended : list of str
        Ranked list of recommended business IDs.
    target_item : str
        The held-out test business ID.
    k : int
        Cutoff rank.

    Returns
    -------
    float
        NDCG@K score.
    """
    topk = recommended[:k]
    if target_item not in topk:
        return 0.0
    rank = topk.index(target_item) + 1
    return 1.0 / math.log2(rank + 1)


def intra_list_diversity_at10(recs, item_feats_df):
    """
    Mean pairwise dissimilarity (1 - cosine similarity) for top-10 recs.

    Higher values imply more heterogeneous recommendations in feature space,
    supporting a hybrid extension analysis beyond pure relevance.

    Parameters
    ----------
    recs : list of str
        Ranked business IDs.
    item_feats_df : pd.DataFrame
        Rows indexed by business_id; feature vectors for cosine geometry.

    Returns
    -------
    float
        Diversity score in [0, 1] for lists with >=2 items having features;
        otherwise 0.0.
    """
    ids = [i for i in recs[:10] if i in item_feats_df.index]
    if len(ids) < 2:
        return 0.0
    X = item_feats_df.loc[ids].values.astype(np.float64)
    sim = cosine_similarity(X)
    n = len(ids)
    off_diag = []
    for i in range(n):
        for j in range(i + 1, n):
            off_diag.append(sim[i, j])
    return float(1.0 - np.mean(off_diag))

## 7. Final Evaluation (Required Deliverable)

**This is the final cell of the notebook.** No training or model fitting occurs after this point.

Evaluation is performed exclusively on the test set. For each test user present in collaborative training (same cohort as standalone CF evaluation), hybrid scores rank candidate restaurants not seen in training; Hit@K / NDCG@K plus extension metrics are reported.

In [8]:
Ks = [10, 20, 30]
hit_sums = {k: 0.0 for k in Ks}
ndcg_sums = {k: 0.0 for k in Ks}
div_sum = 0.0
n_users = 0
union_top10 = set()

for _, row in test_df.iterrows():
    u = str(row['user_id'])
    target = str(row['business_id'])

    if u not in user_idx:
        continue

    seen = train_history.get(u, set())
    recs = get_hybrid_recs(
        u, R_norm, user_idx, item_idx, items,
        item_feats_df, user_profiles, seen, alpha=ALPHA
    )

    for k in Ks:
        hit_sums[k] += hit_at_k(recs, target, k)
        ndcg_sums[k] += ndcg_at_k(recs, target, k)

    div_sum += intra_list_diversity_at10(recs, item_feats_df)
    union_top10.update(recs[:10])
    n_users += 1

coverage10 = len(union_top10) / max(len(items), 1)
mean_div10 = div_sum / max(n_users, 1)

print('Hybrid CF + Content-Based Recommender — Test Metrics')
print('=' * 55)
print(f'Users evaluated: {n_users}')
print(f'Alpha (CF weight): {ALPHA}  |  Content weight: {1-ALPHA}')
print()
print('Ranking Metrics:')
for k in Ks:
    print(f'  Hit@{k}:  {hit_sums[k]/n_users:.4f}    NDCG@{k}:  {ndcg_sums[k]/n_users:.4f}')
print()
print('Extension-specific metrics (hybrid goal):')
print(f'  Catalog coverage @10: {coverage10:.4f}  '
      '(unique items in any user Top-10 / |catalog|)')
print(f'  Mean intra-list diversity @10: {mean_div10:.4f}  '
      '(1 - mean pairwise cosine sim of Top-10 item features)')
print()
print('Re-run standalone collaborative_filtering.ipynb and Content-Based (metadata')
print('only).ipynb to compare Hit@K / NDCG@K on the same test split; numbers above')
print('replace any hard-coded examples from earlier drafts.')

Hybrid CF + Content-Based Recommender — Test Metrics
Users evaluated: 4801
Alpha (CF weight): 0.6  |  Content weight: 0.4

Ranking Metrics:
  Hit@10:  0.0539    NDCG@10:  0.0272
  Hit@20:  0.0969    NDCG@20:  0.0380
  Hit@30:  0.1312    NDCG@30:  0.0454

Extension-specific metrics (hybrid goal):
  Catalog coverage @10: 0.8279  (unique items in any user Top-10 / |catalog|)
  Mean intra-list diversity @10: 0.2174  (1 - mean pairwise cosine sim of Top-10 item features)

Re-run standalone collaborative_filtering.ipynb and Content-Based (metadata
only).ipynb to compare Hit@K / NDCG@K on the same test split; numbers above
replace any hard-coded examples from earlier drafts.
